# Verification

Inspect a saved processing history, reproduce its results, and export the workflow
and recorded environment for reuse.

## Load a recorded result

Complete [DDA-TMT](dda-tmt.ipynb) with the current `msmu` version, then run this
notebook from the same working directory. Its processed result is
`dda_tmt_PXD013361.h5mu`.
You can replace this path with your own recorded analysis.


In [1]:
import msmu as mm

mdata = mm.read_h5mu("dda_tmt_PXD013361.h5mu")

For verified replay, the history must start from recorded file readers,
contain hashes for supported operations that return a `MuData`, and have no gaps
from unrecorded changes. Keep the original input files or URLs available.
See the [replay API](../reference/pv/replay.md) for supported histories, arguments,
and restrictions.
PCA and UMAP require a recorded integer `random_state` (PCA defaults to `0`).

## Inspect the recorded calls

[`mm.pv.get_log()`](../reference/pv/get_log.md) returns a decoded copy of the history.
Read the recorded history:


In [2]:
log = mm.pv.get_log(mdata)

`mm.pv.replay()` and `mm.pv.to_script()` use these events to reconstruct supported
workflows.

## Export and prepare the recorded environment

[`mm.pv.to_env()`](../reference/pv/to_env.md) exports recorded Python and package
versions without changing the data or installing packages:


In [3]:
mm.pv.to_env(mdata, "requirements.txt", format="uv")
mm.pv.to_env(mdata, "environment.yml", format="conda")

WARNING - Recorded MSMU source has uncommitted changes. Package version pins cannot restore them; preserve and install the matching source separately.


WARNING - Recorded MSMU source has uncommitted changes. Package version pins cannot restore them; preserve and install the matching source separately.


An existing destination file is overwritten. Omit the filename to return the
specification as text. Use `environment_id` to select a recorded installation
environment when the history contains more than one:

```python
text = mm.pv.to_env(log, environment_id="<recorded ID>", format="uv")
```

For `uv`, use the Python version printed in the exported file:

```bash
# Replace 3.12.8 with the recorded version printed in requirements.txt.
uv venv --python 3.12.8 .venv-replay
uv pip sync --python .venv-replay requirements.txt
```

For `conda`:

```bash
conda env create --name msmu-replay --file environment.yml
conda activate msmu-replay
```

These files pin Python and package versions; they are not complete environment
lockfiles. Package sources, native libraries, platform differences, and uncommitted
source changes need separate handling. Use the prepared environment and matching
`msmu` version for the replay or script below.

## Replay the workflow

[`mm.pv.replay()`](../reference/pv/replay.md) reruns the recorded calls from their
original inputs and returns a new `MuData` with freshly recorded history:


In [4]:
replayed = mm.pv.replay(mdata)

WARNING - Replay environment may differ: uncommitted source changes in recording. Recorded package versions and source code are not restored automatically.


INFO - Reading SAGE Identification data: 1 file(s)


INFO - Reading SAGE Quantification data: 1 file(s)


INFO - Validating SDRF metadata for https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/dev/data/sage_tmt/meta.sdrf.tsv.


INFO - SDRF validation succeeded for https://raw.githubusercontent.com/bertis-informatics/msmu/refs/heads/dev/data/sage_tmt/meta.sdrf.tsv.


...

INFO - Removed subsumable: 0


INFO - Total protein groups: 1499


INFO - Applying var filters for peptide: ['q_value_lt_0.01', 'peptide_type_eq_unique']


INFO - Protein-level identifications: 1479 (1446 at 1% FDR)


INFO - Applying var filters for protein: ['q_value_lt_0.01']


The supplied object is unchanged. With the default `verify=True`, replay uses the
recorded hashing policy, checks input/output hashes, and stops at the first
mismatch. Environment differences produce warnings; replay does not install
packages.


## Generate a verified script

[`mm.pv.to_script()`](../reference/pv/to_script.md) returns Python source or writes
it to a file without reading the inputs or running the analysis:


In [5]:
script = mm.pv.to_script(mdata)
mm.pv.to_script(mdata, filename="workflow.py")

An existing destination is overwritten after validation. When given a `MuData`,
verified export first checks its current content against the last recorded output;
a mismatch leaves the destination unchanged. Passing a decoded log cannot check
the current data.

Run `workflow.py` in the prepared environment with its original inputs available.
The script checks the environment and recorded hashes using the recorded hashing
policy, then leaves the result in `mdata`. The same supported-workflow restrictions
as replay apply.
